In [1]:
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest
import statsmodels.api as sm
from scipy import stats

In [2]:
from dotenv import load_dotenv

load_dotenv()

from sqlalchemy import create_engine
import os

def connect_to_db():
    engine = create_engine(
        f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@"
        f"{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
    )
    return engine

engine = connect_to_db()

conn = engine.connect()

In [3]:
df = pd.read_sql("SELECT * FROM public.gold_model_dataset", engine)

df.head()

,date_time,user_id,is_mobile,is_package,channel,cnt,trip_type,is_family_trip,is_multi_room,total_guests,...,distance_clean,distance_behavior_group,user_booking_rate,avg_session_intensity,avg_stay_duration,avg_advance_booking_days,mobile_usage_rate,destination_popularity,hotel_cluster,is_booking
0,2014-04-27 19:43:09,14,0,0,9,1,solo,0,0,1,...,2265.9330,far,0.0,1.0,2.0,54.0,0.0,3544,88,0
1,2014-10-22 08:51:12,38,0,0,2,1,group,0,0,2,...,480.7833,medium,0.0,1.0,4.0,123.0,0.0,47,33,0
2,2014-08-05 16:21:41,40,0,0,0,1,solo,0,0,1,...,2576.2959,far,0.0,1.0,1.0,1.0,0.0,100,76,0
3,2014-09-18 22:30:51,156,0,0,5,1,family,1,0,4,...,1128.2299,far,0.0,1.0,1.0,14.0,0.0,214,13,0
4,2014-09-18 22:24:52,156,0,0,5,1,family,1,0,4,...,1127.9475,far,0.0,1.0,1.0,14.0,0.0,214,16,0


In [4]:
group_A = df[df["is_mobile"] == 0]["is_booking"]  # Desktop
group_B = df[df["is_mobile"] == 1]["is_booking"]  # Mobile

In [5]:
print("Desktop:", group_A.mean())
print("Mobile:", group_B.mean())

Desktop: 0.08338264128177462
Mobile: 0.0605512878445549


In [6]:
t_stat, p_value = stats.ttest_ind(group_A, group_B)

print("T-stat:", t_stat)
print("P-value:", p_value)

T-stat: 9.013459370894731
P-value: 2.030772237983561e-19


In [7]:
alpha = 0.05

if p_value < alpha:
    print("Statistically significant difference")
else:
    print("No significant difference")

Statistically significant difference


In [8]:
lift = (group_B.mean() - group_A.mean()) / group_A.mean()
print("Lift:", lift)

Lift: -0.2738142266334047


In [9]:
df[df["is_package"] == 1]["is_booking"]
df[df["is_package"] == 0]["is_booking"]

0        0
1        0
2        0
3        0
4        0
        ..
99465    0
99466    0
99468    0
99469    0
99470    0
Name: is_booking, Length: 75063, dtype: int64

In [10]:
df[df["distance_group"] == "near"]["is_booking"]
df[df["distance_group"] == "far"]["is_booking"]

0        0
2        0
3        0
4        0
5        0
        ..
99461    0
99462    0
99464    0
99466    0
99467    0
Name: is_booking, Length: 34322, dtype: int64

In [11]:
df.groupby(["is_mobile", "trip_type"])["is_booking"].mean()

is_mobile  trip_type
0          family       0.074617
           group        0.072542
           solo         0.125301
1          family       0.051922
           group        0.056544
           solo         0.096279
Name: is_booking, dtype: float64

In [12]:
count = [group_A.sum(), group_B.sum()]
nobs = [len(group_A), len(group_B)]

stat, pval = proportions_ztest(count, nobs)

print(stat, pval)

9.00987127920701 2.062984867632822e-19


In [13]:
print("Desktop:", group_A.mean())
print("Mobile:", group_B.mean())
lift = (group_B.mean() - group_A.mean()) / group_A.mean()
print("Lift:", lift)

Desktop: 0.08338264128177462
Mobile: 0.0605512878445549
Lift: -0.2738142266334047


In [14]:
df.groupby("is_mobile")["is_distance_unknown"].mean()

is_mobile
0    0.362454
1    0.349676
Name: is_distance_unknown, dtype: float64

In [15]:
df.groupby("is_mobile")["cnt"].mean()

is_mobile
0    1.485283
1    1.516268
Name: cnt, dtype: float64

In [16]:
df.groupby("is_mobile")[["cnt", "advance_booking_days", "is_distance_unknown"]].mean()

,cnt,advance_booking_days,is_distance_unknown
is_mobile,,,
0,1.485283,55.031151,0.362454
1,1.516268,56.186097,0.349676


In [17]:
conn.close()

## A/B Test Analysis – Mobile vs Desktop

An observational A/B test was conducted to compare booking rates between mobile and desktop users.

- Desktop booking rate: 8.34%
- Mobile booking rate: 6.06%
- Relative lift: -27.38%

A two-sample statistical test showed a highly significant difference (p < 0.001), indicating that the observed gap is unlikely due to random variation.

### Interpretation

Mobile users exhibit a significantly lower conversion rate compared to desktop users. The effect size is substantial, with mobile showing approximately 27% lower conversion.

Further analysis controlling for key variables such as session intensity (`cnt`), booking window (`advance_booking_days`), and missing distance information (`is_distance_unknown`) indicates that this difference is not explained by these factors.

### Conclusion

The lower conversion rate on mobile appears to be a structural effect, potentially driven by differences in user behavior, context, or user experience across devices.

### Limitations

This analysis is based on observational data and does not represent a randomized controlled experiment. Therefore, causal conclusions cannot be definitively established.